## attribute override

This notebook introduces `attribute :>>` override; after running it you can express named variants of a design by overriding inherited attribute values.

The previous notebook declared that the toaster must complete a cycle in at most 180 seconds. Before checking whether the design meets that requirement, we need to state the operating conditions we are designing for. This notebook introduces `attribute :>>` override: a part usage can redeclare an inherited attribute with a specific value, encoding the assumption being evaluated.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch02-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch02-cumulative.sysml` file adds the first requirements construct: `requirement def TimelyToast` constrains `cycleTime <= 180.0` seconds with a typed `subject` and `require constraint` body. Two candidate parts — `nominal` (default 120 s) and `slow` (overridden to 200 s) — are declared for comparison. The `assert satisfy` pattern comes in Chapter 3; for now the candidates exist without a recorded claim.

In [ ]:
# Negative control: :>> can only override an attribute that already
# exists in the inherited chain. Overriding a non-existent name fails.
bad_source = """
package Bad {
    private import ScalarValues::*;
    part def Toaster { attribute cycleTime : Real default = 120.0; }
    part slow : Toaster {
        attribute :>> nonExistent = 200.0;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
nominal = model.find("ToasterDemo::nominal")
slow = model.find("ToasterDemo::slow")
assert nominal is not None
assert slow is not None

print("nominal:", nominal.id, "| kind:", nominal.kind)
print("slow   :", slow.id, "| kind:", slow.kind)

slow_attrs = slow.attributes()
print(f"slow overridden attributes ({len(slow_attrs)}):")
for a in slow_attrs:
    print(f"  {a.id}")
conn.close()

`part slow : Toaster { attribute :>> cycleTime = 200.0; }` is the A-F override; OpenSysML resolves the redeclaration against the inherited attribute from `Toaster` (O-S); `slow.attributes()` returns the overridden symbol (E).

Try the chapter exercise in `exercises/ch02/exercise.ipynb`: create a `weakBrew` variant of your `CoffeeMaker` with a lower `brewTemp` and confirm the override loads.